Bandit Problems: Epsilon-Greedy Agent

In [ ]:
import numpy as np
import random

class EpsilonGreedyBandit:
    def __init__(self, k=10, epsilon=0.1):
        self.k = k
        self.epsilon = epsilon
        self.q_values = np.zeros(k)
        self.action_counts = np.zeros(k)

    def select_action(self):
        if random.random() < self.epsilon:
            return random.randint(0, self.k - 1)  # Explore
        else:
            return np.argmax(self.q_values)  # Exploit

    def update(self, action, reward):
        self.action_counts[action] += 1
        self.q_values[action] += (reward - self.q_values[action]) / self.action_counts[action]

# Test
bandit = EpsilonGreedyBandit()
for _ in range(100):
    action = bandit.select_action()
    reward = np.random.randn()  # Simulating a random reward
    bandit.update(action, reward)

print("Estimated action values:", bandit.q_values)


Estimated action values: [ 9.83611234e-04 -6.02675265e-01 -8.68874533e-02 -7.27810471e-01
 -2.46536211e+00 -3.52181952e-01 -6.56820729e-01 -3.37899800e-01
 -4.33561244e-01 -2.31473613e-01]


Markov Decision Processes: Episode Returns


In [ ]:
class EpisodeReturns:
    def __init__(self, gamma=0.9):
        self.gamma = gamma

    def compute_return(self, rewards):
        G = 0
        for t in reversed(range(len(rewards))):
            G = rewards[t] + self.gamma * G
        return G

# Test
episode = EpisodeReturns()
rewards = [1, 2, 3, 4]
print("Episode return:", episode.compute_return(rewards))


Episode return: 8.146


Markov Decision Processes: Returns and Discount Factors

In [ ]:
class DiscountedReturns:
    def __init__(self, gamma=0.9):
        self.gamma = gamma

    def compute_discounted_return(self, rewards):
        return sum(reward * (self.gamma ** i) for i, reward in enumerate(rewards))

# Test
discounted = DiscountedReturns()
rewards = [1, 2, 3, 4]
print("Discounted return:", discounted.compute_discounted_return(rewards))


Discounted return: 8.146


The Bellman Equation

In [ ]:
class BellmanEquation:
    def __init__(self, gamma=0.9):
        self.gamma = gamma

    def compute_value(self, reward, next_value):
        return reward + self.gamma * next_value

# Test
bellman = BellmanEquation()
print("Bellman value:", bellman.compute_value(5, 10))


Bellman value: 14.0


Iterative Policy Evaluation and Improvement

In [ ]:
import numpy as np

class PolicyIteration:
    def __init__(self, states, actions, policy, transition_probs, rewards, gamma=0.9):
        self.states = states
        self.actions = actions
        self.policy = policy
        self.transition_probs = transition_probs
        self.rewards = rewards
        self.gamma = gamma
        self.V = {s: 0 for s in states}  # Initialize value function

    def evaluate_policy(self, theta=1e-6):
        while True:
            delta = 0
            new_V = self.V.copy()  # Avoid modifying while iterating
            for s in self.states:
                v = self.V[s]
                new_V[s] = sum(
                    self.policy.get(s, {}).get(a, 0) * sum(
                        self.transition_probs.get(s, {}).get(a, {}).get(s_, 0) *
                        (self.rewards.get(s, {}).get(a, {}).get(s_, 0) + self.gamma * self.V.get(s_, 0))
                        for s_ in self.states
                    ) for a in self.actions
                )
                delta = max(delta, abs(v - new_V[s]))
            self.V = new_V
            if delta < theta:
                break

# Test
states = [0, 1]
actions = [0, 1]
policy = {0: {0: 0.5, 1: 0.5}, 1: {0: 0.7, 1: 0.3}}
transition_probs = {
    0: {0: {0: 0.8, 1: 0.2}, 1: {0: 0.5, 1: 0.5}},
    1: {0: {0: 0.6, 1: 0.4}, 1: {0: 0.3, 1: 0.7}}
}
rewards = {
    0: {0: {0: 5, 1: 10}, 1: {0: 3, 1: 7}},
    1: {0: {0: 2, 1: 8}, 1: {0: 6, 1: 9}}
}

policy_iter = PolicyIteration(states, actions, policy, transition_probs, rewards)
policy_iter.evaluate_policy()
print("Policy values:", policy_iter.V)


Policy values: {0: 55.03603281815081, 1: 55.04747446574805}


Dynamic Programming

In [ ]:
import numpy as np

class DynamicProgramming:
    def __init__(self, states, actions, transition_probs, rewards, gamma=0.9):
        self.states = states
        self.actions = actions
        self.transition_probs = transition_probs
        self.rewards = rewards
        self.gamma = gamma
        self.V = {s: 0 for s in states}  # Use a dictionary to store values per state

    def value_iteration(self, theta=1e-6):
        while True:
            delta = 0
            new_V = self.V.copy()  # Create a copy to update values safely
            for s in self.states:
                v = self.V[s]
                new_V[s] = max(
                    sum(
                        self.transition_probs.get(s, {}).get(a, {}).get(s_, 0) *
                        (self.rewards.get(s, {}).get(a, {}).get(s_, 0) + self.gamma * self.V.get(s_, 0))
                        for s_ in self.states
                    ) for a in self.actions
                )
                delta = max(delta, abs(v - new_V[s]))
            self.V = new_V  # Update values
            if delta < theta:
                break

# Test
states = [0, 1]
actions = [0, 1]
transition_probs = {
    0: {0: {0: 0.8, 1: 0.2}, 1: {0: 0.5, 1: 0.5}},
    1: {0: {0: 0.6, 1: 0.4}, 1: {0: 0.3, 1: 0.7}}
}
rewards = {
    0: {0: {0: 5, 1: 10}, 1: {0: 3, 1: 7}},
    1: {0: {0: 2, 1: 8}, 1: {0: 6, 1: 9}}
}

dp = DynamicProgramming(states, actions, transition_probs, rewards)
dp.value_iteration()
print("Optimal values:", dp.V)


Optimal values: {0: 67.01218666742045, 1: 70.7926744722985}


Q-Learning and Sampling-Based Methods

In [ ]:
class QLearning:
    def __init__(self, states, actions, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.states = states
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((len(states), len(actions)))

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.actions)  # Explore
        else:
            return np.argmax(self.Q[state])  # Exploit

    def update(self, state, action, reward, next_state):
        best_next_action = np.argmax(self.Q[next_state])
        self.Q[state, action] += self.alpha * (reward + self.gamma * self.Q[next_state, best_next_action] - self.Q[state, action])

# Test
ql = QLearning(states, actions)
for _ in range(100):
    state = random.choice(states)
    action = ql.select_action(state)
    reward = np.random.randint(1, 10)
    next_state = random.choice(states)
    ql.update(state, action, reward, next_state)

print("Q-values:\n", ql.Q)


Q-values:
 [[19.69237139  0.8783854 ]
 [17.78629177  3.25362703]]


##Temporal Difference

In [ ]:
# prompt: make me a code for temporal difference in python

import numpy as np

class TemporalDifference:
    def __init__(self, states, actions, alpha=0.1, gamma=0.9):
        self.states = states
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.V = np.zeros(len(states))  # Initialize state values

    def update(self, state, reward, next_state):
        self.V[state] += self.alpha * (reward + self.gamma * self.V[next_state] - self.V[state])

# Example usage
states = [0, 1, 2]  # Example states
actions = [0, 1]  # Example actions
td_learner = TemporalDifference(states, actions)


# Simulate a sequence of state transitions and rewards
# Replace with actual state transitions from your environment
state_transitions = [(0, 1, 5), (1, 2, 10), (2, 1, -5), (1,0, 2)]

for current_state, next_state, reward in state_transitions:
    td_learner.update(current_state, reward, next_state)


print("State Values:", td_learner.V)


State Values: [ 0.5    1.145 -0.41 ]


##Monte Carlo


In [ ]:
# prompt: code for monte carlo in rl using python

import numpy as np
import random

class MonteCarlo:
    def __init__(self, states, actions, gamma=0.9, epsilon=0.1):
        self.states = states
        self.actions = actions
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((len(states), len(actions)))
        self.returns = {}  # Store returns for state-action pairs

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        else:
            return np.argmax(self.Q[state])

    def update(self, episode):
        G = 0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = self.gamma * G + reward
            if (state, action) not in [(x[0], x[1]) for x in episode[:t]]:  # First visit MC
                if (state, action) not in self.returns:
                    self.returns[(state, action)] = []
                self.returns[(state, action)].append(G)
                self.Q[state, action] = np.mean(self.returns[(state, action)])


# Example usage
states = [0, 1, 2]
actions = [0, 1]
mc = MonteCarlo(states, actions)

# Simulate episodes (replace with your environment)
episodes = [
    [ (0, 0, 10), (1, 1, 5), (2, 0, -2)],
    [ (0, 1, 2), (1,0, 8), (2,1, 3)]
]

for episode in episodes:
  mc.update(episode)

print("Q-values:\n", mc.Q)


Q-values:
 [[12.88 11.63]
 [10.7   3.2 ]
 [-2.    3.  ]]
